# Machine Learning Notes — Data Transformation & Feature Engineering

# 1. Scikit-Learn Transformers Output NumPy Arrays

Most Scikit-Learn transformers:

* Take Pandas DataFrames as input
* Return NumPy arrays as output

Example:

```python
X = imputer.transform(housing_num)
```

Even though `housing_num` is a DataFrame,
`X` becomes:

```python
numpy.ndarray
```

---

# Problem With NumPy Arrays

After transformation:

* Column names disappear
* Index disappears

This makes debugging difficult.

---

# Solution: Convert Back to DataFrame

```python
housing_tr = pd.DataFrame(
    X,
    columns=housing_num.columns,
    index=housing_num.index
)
```

---

# Why This Is Important

This restores:

* Feature names
* Row indices
* Better readability
* Easier debugging

---

# 2. Handling Text & Categorical Attributes

Most ML algorithms work only with:

# Numerical data

But datasets often contain:

* Text data
* Categories
* Labels

Example:

```python
ocean_proximity
```

Values:

```text
NEAR BAY
INLAND
<1H OCEAN
```

These are:

# Categorical Features

---

# 3. Types of Categorical Data

## Ordinal Categories

Have order.

Example:

```text
Bad < Average < Good < Excellent
```

---

## Nominal Categories

No order.

Example:

```text
Red, Blue, Green
INLAND, NEAR BAY
```

`ocean_proximity` is:

# Nominal Data

---

# 4. OrdinalEncoder

Converts categories into integers.

---

## Example

```python
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()

housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
```

---

# Output

```python
array([
 [3.],
 [0.],
 [1.]
])
```

Meaning:

| Category     | Number |
| ------------ | ------ |
| `<1H OCEAN`  | 0      |
| `INLAND`     | 1      |
| `ISLAND`     | 2      |
| `NEAR BAY`   | 3      |
| `NEAR OCEAN` | 4      |

---

# Problem With Ordinal Encoding

ML algorithms assume:

```text
4 > 3 > 2 > 1 > 0
```

Meaning:

* Category 4 is “greater”
* Category 0 and 1 are “closer”

But for ocean categories:

# This makes no sense

Example:

```text
INLAND is not mathematically smaller than NEAR BAY
```

This creates:

# False relationships

---

# 5. One-Hot Encoding

Solution:

# Create one binary column per category

---

# Example

Instead of:

```text
NEAR BAY = 3
```

Use:

| <1H OCEAN | INLAND | ISLAND | NEAR BAY | NEAR OCEAN |
| --------- | ------ | ------ | -------- | ---------- |
| 0         | 0      | 0      | 1        | 0          |

Only one column becomes:

# 1 (hot)

Others:

# 0 (cold)

---

# 6. OneHotEncoder

```python
from sklearn.preprocessing import OneHotEncoder

cat_encoder = OneHotEncoder()

housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
```

---

# Output Type

By default:

# Sparse Matrix

---

# 7. Sparse Matrix

A sparse matrix stores:

* Only nonzero values
* Their positions

Useful because one-hot matrices contain:

# Mostly zeros

---

# Example

Instead of storing:

```text
0 0 0 1 0
```

It stores only:

```text
Position of 1
```

---

# Benefits

## Saves Memory

Huge benefit for datasets with:

* Hundreds
* Thousands
* Millions of categories

---

## Faster Computation

Many ML algorithms optimize sparse operations.

---

# 8. Dense vs Sparse Matrix

## Sparse Matrix

Efficient storage.

Example:

```python
<Compressed Sparse Row matrix>
```

---

## Dense Matrix

Regular NumPy array.

Convert using:

```python
housing_cat_1hot.toarray()
```

---

# Force Dense Output

```python
OneHotEncoder(sparse_output=False)
```

---

# 9. Categories Learned by Encoder

```python
cat_encoder.categories_
```

Output:

```python
['<1H OCEAN', 'INLAND', 'ISLAND', ...]
```

---

# 10. Pandas get_dummies()

Alternative method:

```python
pd.get_dummies(df)
```

Example:

```python
pd.get_dummies(df_test)
```

---

# Difference Between get_dummies() and OneHotEncoder

| get_dummies()                         | OneHotEncoder         |
| ------------------------------------- | --------------------- |
| Simpler                               | Production-ready      |
| Does not remember training categories | Remembers categories  |
| Can create mismatched columns         | Maintains consistency |

---

# Why OneHotEncoder Is Better

In production:

* Model expects exact same columns

If new categories appear:

* `get_dummies()` changes columns
* Model breaks

---

# 11. Unknown Categories

Example:

```text
<2H OCEAN
```

Not seen during training.

---

# get_dummies() Behavior

Creates new unexpected column.

Bad for production systems.

---

# OneHotEncoder Behavior

Raises error by default.

Safer.

---

# Ignore Unknown Categories

```python
OneHotEncoder(handle_unknown="ignore")
```

Unknown category becomes:

```text
0 0 0 0 0
```

---

# 12. feature_names_in_

Scikit-Learn stores original column names.

```python
cat_encoder.feature_names_in_
```

Useful for:

* Debugging
* Validation
* Preventing column mismatch

---

# 13. get_feature_names_out()

Returns transformed column names.

```python
cat_encoder.get_feature_names_out()
```

Example:

```text
ocean_proximity_INLAND
ocean_proximity_NEAR BAY
```

---

# Why Important

After one-hot encoding:

* One column becomes many columns

This method tracks new feature names.

---

# 14. Feature Scaling

One of the MOST IMPORTANT preprocessing steps.

---

# Why Scaling Matters

Features may have very different ranges.

Example:

| Feature       | Range      |
| ------------- | ---------- |
| total_rooms   | 6 → 39,320 |
| median_income | 0 → 15     |

Without scaling:

* Large-value features dominate
* Model becomes biased

---

# Algorithms Sensitive to Scaling

Very sensitive:

* Linear Regression
* Logistic Regression
* KNN
* SVM
* Neural Networks
* Gradient Descent

Less sensitive:

* Decision Trees
* Random Forests

---

# 15. Important Rule

# NEVER fit scalers on test data

Correct workflow:

```python
scaler.fit(training_data)
scaler.transform(test_data)
```

---

# Why?

Otherwise:

# Data Leakage

Model indirectly learns test data statistics.

---

# 16. Min-Max Scaling

Also called:

# Normalization

---

## Formula

$$x_{scaled}=\frac{x-x_{min}}{x_{max}-x_{min}}$$

---

# Output Range

Usually:

```text
0 → 1
```

Can change range:

```python
feature_range=(-1,1)
```

---

# Example

| Original | Scaled |
| -------- | ------ |
| 10       | 0.0    |
| 20       | 0.5    |
| 30       | 1.0    |

---

# MinMaxScaler

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
```

---

# Problem With Min-Max Scaling

Sensitive to:

# Outliers

One huge value can crush all other values.

---

# 17. Standardization

Most commonly used scaling method.

---

## Formula

$$x_{scaled}=\frac{x-\mu}{\sigma}$$

Where:

* (\mu) = mean
* (\sigma) = standard deviation

---

# Properties

After scaling:

* Mean = 0
* Standard deviation = 1

---

# StandardScaler

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
```

---

# Why Standardization Is Better

Less affected by outliers.

---

# 18. Heavy-Tailed Distribution

A distribution where:

* Extreme values occur frequently

Example:

```text
population
```

---

# Problem

Scaling alone compresses most values.

---

# Solution: Transform Feature First

---

# 19. Log Transformation

Useful for:

# Right-skewed distributions

---

## Formula

$$x_{new}=\log(x)$$

---

# Benefits

* Reduces skewness
* Compresses large values
* Makes distribution more Gaussian

---

# Example

| Original | Log Value |
| -------- | --------- |
| 10       | 2.3       |
| 1000     | 6.9       |

---

# 20. Square Root Transformation

Another skew-reduction technique.

## Formula

$$x_{new}=\sqrt{x}$$

---

# 21. Gaussian Distribution

Also called:

# Normal Distribution

Bell-shaped curve.

ML algorithms prefer:

* Symmetric
* Smooth distributions

---

# 22. Bucketization (Binning)

Convert continuous values into ranges.

Example:

| Income | Category |
| ------ | -------- |
| 0–1.5  | 1        |
| 1.5–3  | 2        |

---

# Why Bucketize?

Helps:

* Simplify patterns
* Handle skewed data
* Handle multimodal data

---

# 23. Multimodal Distribution

Distribution with:

# Multiple peaks

Example:

```text
housing_median_age
```

---

# Why Difficult?

One model may struggle to learn:

* Multiple separate behaviors

---

# Solution

Treat bucket IDs as:

# Categories

Then apply:

# One-Hot Encoding

---

# 24. Radial Basis Function (RBF)

Measures similarity to a point.

---

# Gaussian RBF Formula

$$RBF(x)=e^{-\gamma(x-c)^2}$$

Where:

* (c) = center point
* (\gamma) = controls spread

---

# Intuition

Closer to center:

* Higher similarity

Far away:

* Similarity rapidly decreases

---

# Example

Similarity to age 35:

* Age 35 → high similarity
* Age 10 → low similarity

---

# 25. rbf_kernel()

```python
from sklearn.metrics.pairwise import rbf_kernel
```

Example:

```python
age_simil_35 = rbf_kernel(
    housing[["housing_median_age"]],
    [[35]],
    gamma=0.1
)
```

---

# 26. Transforming Target Labels

Not only inputs,
sometimes outputs need transformation too.

---

# Example

If house prices are heavily skewed:

* Apply log transform

---

# Problem

Model now predicts:

```text
log(price)
```

Not:

```text
price
```

---

# Solution

Use:

# inverse_transform()

---

# 27. inverse_transform()

Converts transformed predictions back to original scale.

Example:

```python
target_scaler.inverse_transform(predictions)
```

---

# 28. Scaling Labels

Example:

```python
target_scaler = StandardScaler()

scaled_labels = target_scaler.fit_transform(
    housing_labels.to_frame()
)
```

---

# Why to_frame()?

Scikit-Learn expects:

# 2D input

Series is:

# 1D

---

# 29. TransformedTargetRegressor

Automates target transformation.

---

# Without It

Manual steps:

1. Scale labels
2. Train model
3. Predict scaled values
4. Inverse transform predictions

Error-prone.

---

# With TransformedTargetRegressor

Everything becomes automatic.

---

# Example

```python
from sklearn.compose import TransformedTargetRegressor

model = TransformedTargetRegressor(
    LinearRegression(),
    transformer=StandardScaler()
)
```

---

# Benefits

## Cleaner Code

Less manual work.

---

## Safer

Avoids scaling mismatch bugs.

---

## Production Friendly

Consistent preprocessing.

---

# 30. Most Important Concepts You Learned

| Concept                    | Meaning                         |
| -------------------------- | ------------------------------- |
| Ordinal Encoding           | Categories → integers           |
| One-Hot Encoding           | Categories → binary columns     |
| Sparse Matrix              | Memory-efficient matrix         |
| Feature Scaling            | Make features comparable        |
| Min-Max Scaling            | Scale to fixed range            |
| Standardization            | Zero mean, unit variance        |
| Heavy Tail                 | Extreme values common           |
| Log Transform              | Reduce skewness                 |
| Bucketization              | Convert values into ranges      |
| RBF                        | Similarity function             |
| inverse_transform          | Return to original scale        |
| TransformedTargetRegressor | Automatic target transformation |

---

# 31. Most Important Practical Lessons

## Always Remember

### NEVER fit preprocessing on test data

Correct:

```python
fit(train)
transform(test)
```

---

### One-Hot Encoding is safer than Ordinal Encoding for nominal data

---

### Scaling is extremely important

Especially for:

* Gradient-based algorithms
* Distance-based algorithms

---

### Transform skewed distributions before scaling

---

### Production ML requires consistent preprocessing

That is why:

* Encoders remember categories
* Scalers remember statistics
* Pipelines are important
